# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from huggingface_hub import login

login()

In [2]:
from google.colab import userdata
import os

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

In [3]:
from datasets import load_dataset
import duckdb


fact_content_ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train")
fact_content = fact_content_ds.data.table   # Arrow, not pandas — much lighter

dim_content_ds = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train")
dim_content = dim_content_ds.data.table

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

...

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/flyrank-internship/work/outputs"
os.makedirs(BASE, exist_ok=True)

Mounted at /content/drive


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

feature_vector = duckdb.sql("""
    WITH prior_position AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position) AS gsc_avg_position_prior
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        d.word_count,
        d.search_volume,
        d.competition_level,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-01') AS content_age_days,
        p.gsc_avg_position_prior
    FROM dim_content d
    LEFT JOIN prior_position p
        USING (client_hash_id, content_hash_id)
    WHERE d.is_published IS TRUE
      AND d.is_deleted IS FALSE
      AND d.content_created_date <= DATE '2026-03-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Missing value handling ---
# word_count: missing is tied to provider_used being NULL (an older ingestion gap),
# not "this page has no words" — so fill with median AND keep a flag, never fill with 0.
feature_vector["word_count_missing"] = feature_vector["word_count"].isna().astype(int)
feature_vector["word_count"] = feature_vector["word_count"].fillna(feature_vector["word_count"].median())

In [7]:
# gsc_avg_position_prior: missing means "no GSC history in February" — a real, structural
# gap (young page or no GSC access), not "position is average." Flag it, then fill.
feature_vector["gsc_position_missing"] = feature_vector["gsc_avg_position_prior"].isna().astype(int)
feature_vector["gsc_avg_position_prior"] = feature_vector["gsc_avg_position_prior"].fillna(
    feature_vector["gsc_avg_position_prior"].median()
)

In [8]:
# --- Categorical handling ---
# competition_level is ordinal (LOW < MEDIUM < HIGH), not unordered — map it to numbers
# rather than one-hot encoding, so the model can use the natural order.
competition_map = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
feature_vector["competition_level_encoded"] = feature_vector["competition_level"].map(competition_map)

feature_vector.head()

,client_hash_id,content_hash_id,word_count,search_volume,competition_level,content_age_days,gsc_avg_position_prior,word_count_missing,gsc_position_missing,competition_level_encoded
0,client_0797ff3a1fc9a6a5,content_9323fd059ff65ad0,3869,30,LOW,145,15.375000,0,0,0.0
1,client_0797ff3a1fc9a6a5,content_9bb9c6a63fc2003f,3950,20,LOW,145,9.195513,0,0,0.0
2,client_0797ff3a1fc9a6a5,content_9d5a482ec16ea617,4067,10,LOW,145,9.181159,0,0,0.0
3,client_0797ff3a1fc9a6a5,content_9ed024c40862c4aa,3763,30,LOW,145,3.000000,0,0,0.0
4,client_0797ff3a1fc9a6a5,content_a0a6b37ae2f9a09c,3075,0,LOW,145,14.690257,0,0,0.0


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Rebuild label: did clicks decline in March vs. February?
labels = duckdb.sql("""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_feb
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    march AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_march
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT feb.client_hash_id, feb.content_hash_id, clicks_feb, clicks_march,
           CASE WHEN clicks_march < clicks_feb THEN 1 ELSE 0 END AS declined
    FROM feb JOIN march USING (client_hash_id, content_hash_id)
""").df()

data = feature_vector.merge(labels, on=["client_hash_id", "content_hash_id"])

honest_cols = ["word_count", "word_count_missing", "search_volume",
               "competition_level_encoded", "content_age_days",
               "gsc_avg_position_prior", "gsc_position_missing"]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
# search_volume: missing means no keyword-volume data was ever recorded for this page.
# Flag it, then fill with median (0 would wrongly imply "definitely no search demand").
data["search_volume_missing"] = data["search_volume"].isna().astype(int)
data["search_volume"] = data["search_volume"].fillna(data["search_volume"].median())

# competition_level_encoded: NaN here means competition_level was NULL or an unmapped
# value (not LOW/MEDIUM/HIGH) — this is categorical, so use -1 as "unknown", not median.
data["competition_level_encoded"] = data["competition_level_encoded"].fillna(-1)

# Confirm all gaps are gone
print(data[honest_cols + ["search_volume_missing"]].isna().sum())

word_count                   0
word_count_missing           0
search_volume                0
competition_level_encoded    0
content_age_days             0
gsc_avg_position_prior       0
gsc_position_missing         0
search_volume_missing        0
dtype: int64


In [11]:
honest_cols = ["word_count", "word_count_missing", "search_volume", "search_volume_missing",
               "competition_level_encoded", "content_age_days",
               "gsc_avg_position_prior", "gsc_position_missing"]

In [12]:
X = data[honest_cols]
y = data["declined"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("Honest AUC:", honest_auc)


Honest AUC: 0.6278150489500833


In [13]:
# Attack 1: raw same-window value
data["clicks_march_test"] = data["clicks_march"]
X_attack1 = data[honest_cols + ["clicks_march_test"]]
X_tr, X_te, y_tr, y_te = train_test_split(X_attack1, y, test_size=0.3, random_state=42)
m1 = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
print("Attack 1 (clicks_march) AUC:", roc_auc_score(y_te, m1.predict_proba(X_te)[:, 1]))

# Attack 2: the actual label-derived column
data["clicks_diff_test"] = data["clicks_march"] - data["clicks_feb"]
X_attack2 = data[honest_cols + ["clicks_diff_test"]]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_attack2, y, test_size=0.3, random_state=42)
m2 = LogisticRegression(max_iter=1000).fit(X_tr2, y_tr2)
print("Attack 2 (clicks_diff) AUC:", roc_auc_score(y_te2, m2.predict_proba(X_te2)[:, 1]))

Attack 1 (clicks_march) AUC: 0.6280079985555984
Attack 2 (clicks_diff) AUC: 1.0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule, in plain words: flag a page as high-priority for refresh if it's `(a)` ranking reasonably well but showing signs of decline `(staleness signal)`, or `(b)` getting far fewer clicks than a page in its position should `(CTR-underperformance signal).` Combine both into one score; whichever signal is stronger for that page becomes the reason code.

**Reason codes this rule can output:**

* `STALE_DECLINING` — older content, position trending down
* `CTR_UNDERPERFORM `— position is fine, but CTR is well below what's typical for that position
* `BOTH `— both signals fire at once (highest priority)
* `MONITOR` — neither signal fires strongly; no action needed yet

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rule_name = "Staleness + CTR-underperformance composite score"
reason_codes = ["STALE_DECLINING", "CTR_UNDERPERFORM", "BOTH", "MONITOR"]
print(rule_name)
print(reason_codes)

Staleness + CTR-underperformance composite score
['STALE_DECLINING', 'CTR_UNDERPERFORM', 'BOTH', 'MONITOR']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

What "bucket" means here:
A bucket is just a group you sort pages into based on a value, so you can compare groups instead of staring at thousands of individual numbers. Like sorting laundry into piles — whites, colors, darks. Here, instead of piles of clothes, you're making piles of pages: "pages younger than 90 days," "pages 90-180 days," "pages older than 180 days." Then you check: does one pile behave differently than the others?

In plain words: "Sort every page into one of three piles by age — young, middle-aged, old. For each pile, count how many pages are in it `(n)`, and calculate what percent of that pile declined `(pct_declined)`."

You end up with a small table like:

age_bucket	   n	     pct_declined
<90d	      40,000	      12%
90-180d	    30,000	      18%
180d+	      20,000	      25%

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# ---- Signal 1: Staleness (content_age_days vs. decline) ----
# Behind the real flag: "refresh flags" in the FlyRank session are staleness-driven.
age_check = duckdb.sql("""
    SELECT
        CASE
            WHEN content_age_days < 90 THEN '<90d'
            WHEN content_age_days < 180 THEN '90-180d'
            ELSE '180d+'
        END AS age_bucket,
        COUNT(*) AS n,
        AVG(declined) AS pct_declined
    FROM (
        SELECT f.content_age_days, l.declined
        FROM feature_vector f
        JOIN labels l USING (client_hash_id, content_hash_id)
    )
    GROUP BY age_bucket
    ORDER BY age_bucket
""").df()
print(age_check)

  age_bucket      n  pct_declined
0      180d+  68735      0.205543
1    90-180d  25785      0.192127
2       <90d  39566      0.134686


In [22]:
# ---- Signal 2: CTR-vs-position (underperformance) ----
# Behind the real flag: "CTR-fix logic" in the session.
# Build actual Feb CTR + a simple expected-CTR-by-position benchmark, then compare.
ctr_check = duckdb.sql("""
    WITH feb_ctr AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS clicks_feb,
               SUM(gsc_impressions) AS impressions_feb,
               AVG(gsc_avg_position) AS position_feb
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) > 0
    )
    SELECT client_hash_id, content_hash_id,
           clicks_feb * 1.0 / impressions_feb AS actual_ctr,
           position_feb,
           -- simple expected CTR benchmark by position bucket
           CASE
               WHEN position_feb <= 3 THEN 0.25
               WHEN position_feb <= 10 THEN 0.05
               ELSE 0.02
           END AS expected_ctr
    FROM feb_ctr
""").df()

ctr_check["ctr_gap"] = ctr_check["actual_ctr"] - ctr_check["expected_ctr"]
ctr_check["underperforming"] = ctr_check["ctr_gap"] < 0

merged_ctr = ctr_check.merge(labels, on=["client_hash_id", "content_hash_id"])
ctr_bucket = merged_ctr.groupby("underperforming")["declined"].agg(["mean", "count"])
print(ctr_bucket)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                     mean   count
underperforming                  
False            0.761141    2244
True             0.172349  131994


In [23]:
expected_ctr_map = {
    "top3": 0.0106,
    "top10": 0.0052,
    "below10": 0.0028
}

ctr_check = duckdb.sql("""
    SELECT client_hash_id, content_hash_id,
           clicks_feb * 1.0 / impressions_feb AS actual_ctr,
           position_feb
    FROM feb_ctr
""").df()

def get_expected(pos):
    if pos <= 3:
        return expected_ctr_map["top3"]
    elif pos <= 10:
        return expected_ctr_map["top10"]
    else:
        return expected_ctr_map["below10"]

ctr_check["expected_ctr"] = ctr_check["position_feb"].apply(get_expected)
ctr_check["ctr_gap"] = ctr_check["actual_ctr"] - ctr_check["expected_ctr"]
ctr_check["underperforming"] = ctr_check["ctr_gap"] < 0

merged_ctr = ctr_check.merge(labels, on=["client_hash_id", "content_hash_id"])
ctr_bucket = merged_ctr.groupby("underperforming")["declined"].agg(["mean", "count"])
print(ctr_bucket)

                     mean   count
underperforming                  
False            0.558650   18670
True             0.121374  115568


In [20]:
feb_ctr = duckdb.sql("""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) AS clicks_feb,
           SUM(gsc_impressions) AS impressions_feb,
           AVG(gsc_avg_position) AS position_feb
    FROM fact_content
    WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

# now feb_ctr is a real DataFrame DuckDB can query by name
duckdb.sql("""
    SELECT
        CASE WHEN position_feb <= 3 THEN 'top3'
             WHEN position_feb <= 10 THEN 'top10'
             ELSE 'below10' END AS pos_bucket,
        MEDIAN(clicks_feb * 1.0 / impressions_feb) AS median_ctr,
        COUNT(*) AS n
    FROM feb_ctr
    GROUP BY pos_bucket
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬───────┐
│ pos_bucket │ median_ctr │   n   │
│  varchar   │   double   │ int64 │
├────────────┼────────────┼───────┤
│ top10      │        0.0 │ 75898 │
│ top3       │        0.0 │ 19243 │
│ below10    │        0.0 │ 58418 │
└────────────┴────────────┴───────┘

In [24]:
duckdb.sql("""
    SELECT
        AVG(CASE WHEN clicks_feb = 0 THEN 1.0 ELSE 0 END) AS pct_zero_clicks,
        AVG(CASE WHEN impressions_feb = 0 THEN 1.0 ELSE 0 END) AS pct_zero_impressions,
        AVG(clicks_feb) AS avg_clicks,
        AVG(impressions_feb) AS avg_impressions
    FROM feb_ctr
""")

┌────────────────────┬──────────────────────┬────────────────────┬────────────────────┐
│  pct_zero_clicks   │ pct_zero_impressions │     avg_clicks     │  avg_impressions   │
│       double       │        double        │       double       │       double       │
├────────────────────┼──────────────────────┼────────────────────┼────────────────────┤
│ 0.6412453845101883 │                  0.0 │ 3.8174577849556197 │ 1173.0274487330603 │
└────────────────────┴──────────────────────┴────────────────────┴────────────────────┘

In [25]:
duckdb.sql("""
    SELECT
        CASE WHEN position_feb <= 3 THEN 'top3'
             WHEN position_feb <= 10 THEN 'top10'
             ELSE 'below10' END AS pos_bucket,
        AVG(clicks_feb * 1.0 / impressions_feb) AS mean_ctr,
        COUNT(*) AS n
    FROM feb_ctr
    GROUP BY pos_bucket
""")

┌────────────┬──────────────────────┬───────┐
│ pos_bucket │       mean_ctr       │   n   │
│  varchar   │        double        │ int64 │
├────────────┼──────────────────────┼───────┤
│ top10      │ 0.005194634309476088 │ 75898 │
│ top3       │  0.01059800284007321 │ 19243 │
│ below10    │ 0.002807781504693828 │ 58418 │
└────────────┴──────────────────────┴───────┘

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [26]:
import os

# Merge staleness features with the CTR check (using real, data-derived expected_ctr)
scored = feature_vector.merge(
    ctr_check[["client_hash_id", "content_hash_id", "ctr_gap", "underperforming", "actual_ctr"]],
    on=["client_hash_id", "content_hash_id"], how="left"
)

# --- Staleness score (CONFIRMED signal — use as-is) ---
scored["staleness_score"] = (scored["content_age_days"] / scored["content_age_days"].max()).clip(0, 1)

# --- CTR score (corrected: only apply to pages with meaningful existing click volume,
#     since low-volume pages showed a floor effect — they can't "decline" much further,
#     which made naive CTR-underperformance look backwards in the signal check) ---
has_volume = scored["actual_ctr"].notna() & (scored["actual_ctr"] > scored["actual_ctr"].median())

scored["ctr_score"] = 0.0
scored.loc[has_volume, "ctr_score"] = (-scored.loc[has_volume, "ctr_gap"]).clip(lower=0)
scored["ctr_score"] = (scored["ctr_score"] / scored["ctr_score"].max()).fillna(0)

# --- Combined score: staleness weighted higher since it was the cleaner, CONFIRMED signal ---
scored["score"] = 0.7 * scored["staleness_score"] + 0.3 * scored["ctr_score"]

# --- Reason codes + actions ---
def reason_code(row):
    stale = row["staleness_score"] > 0.5
    ctr_bad = row["ctr_score"] > 0.5
    if stale and ctr_bad:
        return "BOTH"
    elif stale:
        return "STALE_DECLINING"
    elif ctr_bad:
        return "CTR_UNDERPERFORM"
    return "MONITOR"

scored["reason_code"] = scored.apply(reason_code, axis=1)
scored["action"] = scored["reason_code"].map({
    "BOTH": "refresh_now",
    "STALE_DECLINING": "refresh",
    "CTR_UNDERPERFORM": "fix_snippet_or_title",
    "MONITOR": "monitor"
})

# --- Rank and save ---
ranked = scored.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
ranked[["client_hash_id", "content_hash_id", "score", "reason_code", "action"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False
)
print("Saved", len(ranked), "rows.")
ranked.head(10)

Saved 303332 rows.


,client_hash_id,content_hash_id,word_count,search_volume,competition_level,content_age_days,gsc_avg_position_prior,word_count_missing,gsc_position_missing,competition_level_encoded,ctr_gap,underperforming,actual_ctr,staleness_score,ctr_score,score,reason_code,action
0,client_e547b89c05043229,content_3eb93e7a5bbd5222,3218,6600,LOW,445,0.371457,0,0,0.0,-0.009757,True,0.000843,0.959052,0.920933,0.947616,BOTH,refresh_now
1,client_e547b89c05043229,content_63f49adb25b4d191,2986,170,LOW,445,2.968058,0,0,0.0,-0.009094,True,0.001506,0.959052,0.858336,0.928837,BOTH,refresh_now
2,client_e547b89c05043229,content_319dd6ecf1f4979b,2907,<NA>,None,437,1.565239,0,0,NaN,-0.009322,True,0.001278,0.941810,0.879901,0.923238,BOTH,refresh_now
3,client_e547b89c05043229,content_0b7307b15de7a9bc,3172,140,LOW,445,2.968126,0,0,0.0,-0.008346,True,0.002254,0.959052,0.787744,0.907659,BOTH,refresh_now
4,client_e547b89c05043229,content_f43b95944a9fa212,2330,30,LOW,437,2.318856,1,0,0.0,-0.008486,True,0.002114,0.941810,0.800948,0.899552,BOTH,refresh_now
5,client_e547b89c05043229,content_878dff2b9c62d9fa,2469,2400,LOW,437,1.620812,0,0,0.0,-0.008392,True,0.002208,0.941810,0.792127,0.896905,BOTH,refresh_now
6,client_e547b89c05043229,content_07b9d4e5f1732a37,2330,0,LOW,404,0.917710,1,0,0.0,-0.009730,True,0.000870,0.870690,0.918408,0.885005,BOTH,refresh_now
7,client_e547b89c05043229,content_53e6c76d677d4498,2330,0,LOW,404,1.040953,1,0,0.0,-0.009613,True,0.000987,0.870690,0.907370,0.881694,BOTH,refresh_now
8,client_e547b89c05043229,content_74199244facea367,2330,0,LOW,404,1.326357,1,0,0.0,-0.009334,True,0.001266,0.870690,0.881008,0.873785,BOTH,refresh_now
9,client_e547b89c05043229,content_c07b4968e3fc4779,3102,40,LOW,387,2.631950,0,0,0.0,-0.010112,True,0.000488,0.834052,0.954424,0.870163,BOTH,refresh_now


In [27]:
ranked.head(20)[["client_hash_id", "content_hash_id", "score", "reason_code", "action",
                  "content_age_days", "actual_ctr", "ctr_gap"]]

,client_hash_id,content_hash_id,score,reason_code,action,content_age_days,actual_ctr,ctr_gap
0,client_e547b89c05043229,content_3eb93e7a5bbd5222,0.947616,BOTH,refresh_now,445,0.000843,-0.009757
1,client_e547b89c05043229,content_63f49adb25b4d191,0.928837,BOTH,refresh_now,445,0.001506,-0.009094
2,client_e547b89c05043229,content_319dd6ecf1f4979b,0.923238,BOTH,refresh_now,437,0.001278,-0.009322
3,client_e547b89c05043229,content_0b7307b15de7a9bc,0.907659,BOTH,refresh_now,445,0.002254,-0.008346
4,client_e547b89c05043229,content_f43b95944a9fa212,0.899552,BOTH,refresh_now,437,0.002114,-0.008486
5,client_e547b89c05043229,content_878dff2b9c62d9fa,0.896905,BOTH,refresh_now,437,0.002208,-0.008392
6,client_e547b89c05043229,content_07b9d4e5f1732a37,0.885005,BOTH,refresh_now,404,0.000870,-0.009730
7,client_e547b89c05043229,content_53e6c76d677d4498,0.881694,BOTH,refresh_now,404,0.000987,-0.009613
8,client_e547b89c05043229,content_74199244facea367,0.873785,BOTH,refresh_now,404,0.001266,-0.009334
9,client_e547b89c05043229,content_c07b4968e3fc4779,0.870163,BOTH,refresh_now,387,0.000488,-0.010112


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [28]:
# Pages where score is high but for a reason that seems shaky
weak = ranked[(ranked["reason_code"] == "CTR_UNDERPERFORM")].head(10)
weak

,client_hash_id,content_hash_id,word_count,search_volume,competition_level,content_age_days,gsc_avg_position_prior,word_count_missing,gsc_position_missing,competition_level_encoded,ctr_gap,underperforming,actual_ctr,staleness_score,ctr_score,score,reason_code,action
8175,client_73cda7b4e4f265ea,content_8a7032104ca97ed6,2330,0,LOW,230,1.649079,1,0,0.0,-0.010367,True,0.000233,0.495690,0.978456,0.640519,CTR_UNDERPERFORM,fix_snippet_or_title
8227,client_73cda7b4e4f265ea,content_ff7a826ad2d639b6,2882,0,LOW,230,1.081746,0,0,0.0,-0.010321,True,0.000279,0.495690,0.974118,0.639218,CTR_UNDERPERFORM,fix_snippet_or_title
8289,client_73cda7b4e4f265ea,content_29fc14cc164f00be,2330,0,LOW,230,1.929083,1,0,0.0,-0.010261,True,0.000339,0.495690,0.968466,0.637522,CTR_UNDERPERFORM,fix_snippet_or_title
8343,client_73cda7b4e4f265ea,content_702fcc3226d82233,2330,0,LOW,230,2.554229,1,0,0.0,-0.010206,True,0.000394,0.495690,0.963316,0.635977,CTR_UNDERPERFORM,fix_snippet_or_title
8354,client_e547b89c05043229,content_35f10273f0baf406,2505,40,LOW,226,1.975194,0,0,0.0,-0.010415,True,0.000185,0.487069,0.983037,0.635859,CTR_UNDERPERFORM,fix_snippet_or_title
8356,client_73cda7b4e4f265ea,content_b99be1e69826799f,2330,20,LOW,229,1.413718,1,0,0.0,-0.010253,True,0.000347,0.493534,0.967687,0.635780,CTR_UNDERPERFORM,fix_snippet_or_title
8426,client_73cda7b4e4f265ea,content_0a7a764302e3eb43,2330,0,LOW,230,2.402152,1,0,0.0,-0.010144,True,0.000456,0.495690,0.957424,0.634210,CTR_UNDERPERFORM,fix_snippet_or_title
8431,client_73cda7b4e4f265ea,content_aa2e2c72b26a55d0,2330,0,LOW,230,2.697514,1,0,0.0,-0.010137,True,0.000463,0.495690,0.956826,0.634031,CTR_UNDERPERFORM,fix_snippet_or_title
8474,client_73cda7b4e4f265ea,content_aaccff67e3244af4,2330,20,LOW,229,2.289849,1,0,0.0,-0.010151,True,0.000449,0.493534,0.958138,0.632916,CTR_UNDERPERFORM,fix_snippet_or_title
8496,client_73cda7b4e4f265ea,content_c5dba0d9608dfcf2,2330,50,HIGH,229,1.458253,1,0,2.0,-0.010123,True,0.000477,0.493534,0.955451,0.632110,CTR_UNDERPERFORM,fix_snippet_or_title


In [29]:
leak_check_cols = ["clicks_march", "clicks_diff", "is_deleted", "is_published", "gsc_data_available"]
present = [c for c in leak_check_cols if c in scored.columns]
print("Excluded/leaky columns present in final scored frame:", present)
# expect: []

Excluded/leaky columns present in final scored frame: []


In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked.head(20)[["client_hash_id", "content_hash_id", "score", "reason_code", "action"]]
top20

,client_hash_id,content_hash_id,score,reason_code,action
0,client_e547b89c05043229,content_3eb93e7a5bbd5222,0.947616,BOTH,refresh_now
1,client_e547b89c05043229,content_63f49adb25b4d191,0.928837,BOTH,refresh_now
2,client_e547b89c05043229,content_319dd6ecf1f4979b,0.923238,BOTH,refresh_now
3,client_e547b89c05043229,content_0b7307b15de7a9bc,0.907659,BOTH,refresh_now
4,client_e547b89c05043229,content_f43b95944a9fa212,0.899552,BOTH,refresh_now
5,client_e547b89c05043229,content_878dff2b9c62d9fa,0.896905,BOTH,refresh_now
6,client_e547b89c05043229,content_07b9d4e5f1732a37,0.885005,BOTH,refresh_now
7,client_e547b89c05043229,content_53e6c76d677d4498,0.881694,BOTH,refresh_now
8,client_e547b89c05043229,content_74199244facea367,0.873785,BOTH,refresh_now
9,client_e547b89c05043229,content_c07b4968e3fc4779,0.870163,BOTH,refresh_now


In [31]:
# Leakage check — confirm nothing from the label window or excluded flags made it into the score
leak_check_cols = ["clicks_march", "clicks_diff", "is_deleted", "is_published", "gsc_data_available"]
present = [c for c in leak_check_cols if c in scored.columns]
print("Excluded/leaky columns present in final scored frame:", present)
# expect: []

Excluded/leaky columns present in final scored frame: []


In [32]:
# Week 3 notebook — save what Week 4+ will need
feature_vector.to_csv(f"{BASE}/feature_vector.csv", index=False)
labels.to_csv(f"{BASE}/labels.csv", index=False)

In [33]:
# Week 4 notebook — save what Week 5+ (and this signal-audit notebook) will need
ctr_check.to_csv(f"{BASE}/ctr_check.csv", index=False)
ranked.to_csv(f"{BASE}/ranked_baseline.csv", index=False)


---

## Week 4 Summary — Building My First Baseline Rule

**What I was trying to do:** turn my content-refresh idea into an actual working rule — something that looks at every page, gives it a score, says *why* it got that score, and tells someone what to do about it. No machine learning yet — just an honest, checkable formula.

**Step 1: Test my ideas before trusting them.**
Before building anything, I checked whether my two signals actually show up in the real data — I didn't just assume they mattered.

- **Staleness (page age):** I split pages into buckets by age (under 90 days, 90-180 days, 180+ days) and checked how often each bucket declined. Older pages declined more often, in a clear, steady pattern (13.5% → 19.2% → 20.6%). **This one held up — CONFIRMED.**

- **CTR vs. position:** My first attempt used a guessed benchmark for "expected CTR" and it badly broke — almost every page got flagged as "underperforming," which made the check meaningless. I fixed it by calculating the *real* average CTR for each position group from the actual data. Once fixed, I found something surprising: pages with a *worse* CTR actually declined *less* than pages with a good CTR. This looked backwards at first, but it makes sense once you think about it — a page that's already barely getting clicks has almost nowhere left to fall. This is called a **floor effect** — something already near zero can't "decline" much further, simply because of the math, not because it's healthy. So I didn't throw the signal away — I fixed the rule to only use CTR as a warning sign for pages that already have real, meaningful traffic.

**Why finding a "wrong" result is actually valuable:** if I had built my rule around the wrong version of the CTR idea, it would have quietly flagged the wrong pages every week without anyone noticing. Catching it now, with a small test, saved the whole rule from being backwards.

**Step 2: Turn the two checked signals into one rule.**
I combined staleness and (fixed) CTR into a single score per page, weighting staleness higher (70%) since it was the stronger, cleaner signal, and CTR lower (30%) since it needed more caution. Every page also gets:
- A **reason code** — one word explaining *why* it scored the way it did (`STALE_DECLINING`, `CTR_UNDERPERFORM`, `BOTH`, or `MONITOR`)
- An **action** — what to actually do about it (refresh, fix the title/snippet, or just keep watching)

I sorted every page by score, highest first, and saved that full ranked list to a CSV file — that list is the actual deliverable: a priority queue someone could work through instead of guessing.

**Step 3: Check the leakage rule still holds.**
I confirmed none of the columns from the future (like March's actual click numbers) or internal product flags (like "is_deleted") snuck into the score. The rule only uses information that would have been known *before* the decision date — otherwise it would look great here and fail completely in real use, since March hasn't happened yet when this rule would actually run.

**Key terms I used this week, in plain words:**
- **Bucket** — a group of pages sorted by some value (like age), so you can compare groups instead of staring at every row.
- **Verdict (CONFIRMED / OPPOSITE / MIXED / FALSE)** — a one-word honest judgment on whether my belief about a signal actually matched the data.
- **Floor effect** — when something is already so low it can't realistically go much lower, which can make a signal look "backwards" even though it's not actually wrong, just limited.
- **Reason code** — a short label explaining *why* a page got flagged, not just that it did.
- **Ranked queue** — the full list of pages, sorted so the most urgent ones are at the top.
- **Leakage check** — confirming the rule never secretly used information from after the decision date, which would make it fail in the real world even if it looks perfect in testing.

**The main lesson of the week:** a rule is only trustworthy if you've actually tested the beliefs behind it. My CTR idea would have been backwards if I'd shipped it as originally planned — checking it first, understanding *why* it was backwards, and fixing the rule accordingly is what makes this a defensible baseline instead of just a guess dressed up as a formula.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.